# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load, overview, and analyze the FAIR² dataset using the `mlcroissant` library, referencing all data components by their `@id` fields, and following the Croissant schema. 

### Dataset Source
The dataset schema is provided as a Croissant JSON-LD file:

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
ds = mlc.Dataset(croissant_url)
meta = ds.metadata
print("Dataset Title: ", meta.name)
print("Description: ", meta.description)

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

In [ ]:
# List all available record sets and fields by @id
print("Record sets found in dataset schema:")
rs_list = []
for record_set in ds.record_sets():
    print(f"  Record set name: {record_set.name!r}")
    print(f"    @id: {record_set.id}")
    rs_list.append(record_set.id)
    if hasattr(record_set, 'fields'):
        print("    Fields:")
        for field in record_set.fields:
            print(f"      Field name: {field.name!r}  @id: {field.id}")
        print("")

## 3. Data Extraction
Load data from the selected record set(s) using their `@id`. For each record set, data is loaded into a DataFrame. Replace `<record_set_id>` and `<field_id>` in later steps using the printed `@id` values from above.

In [ ]:
# Automatically extract all record set @id values from overview cell above
record_set_ids = rs_list
print(f"Found record sets with @id:", record_set_ids)

# Load each record set into a dataframe
dataframes = {}
for record_set_id in record_set_ids:
    print(f'Loading data from record set @id={record_set_id} …')
    records = list(ds.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Loaded {len(df)} rows, columns: {list(df.columns)}")

# As an example, select the first available record set for display:
example_record_set_id = record_set_ids[0] if record_set_ids else None
if example_record_set_id:
    print(f"\nFirst 5 records of record set {example_record_set_id}:")
    display(dataframes[example_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data filtering, normalization, and grouping. Please update `<numeric_field_id>` and `<group_field_id>` with valid column `@id`s printed above for meaningful results.

In [ ]:
# For illustration, we choose the first available numeric column.
# Please update numeric_field_id and group_field_id as appropriate for the dataset content.

from pandas.api.types import is_numeric_dtype

if example_record_set_id:
    df = dataframes[example_record_set_id]
    # Attempt to automatically select a numeric field
    numeric_field_id = None
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is not None:
        # Filtering records (demo threshold = 10)
        threshold = 10
        fdf = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records from {numeric_field_id} > {threshold}:")
        display(fdf.head())

        # Normalization
        fdf[f"{numeric_field_id}_normalized"] = (fdf[numeric_field_id] - fdf[numeric_field_id].mean()) / fdf[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(fdf[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a non-numeric field (pick the first suitable one)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and not is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped = fdf.groupby(group_field_id, observed=False)[numeric_field_id].mean()
            print(f"Mean {numeric_field_id} by {group_field_id}:")
            display(grouped.head())
        else:
            print("No suitable group field was found for grouping.")
    else:
        print("No numeric field for EDA in this record set.")
else:
    print("No data to analyze.")

## 5. Visualization
Visualize data distributions and relationships in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example visualization: Histogram of the selected numeric field
if example_record_set_id and numeric_field_id:
    plt.figure(figsize=(7, 4))
    df = dataframes[example_record_set_id]
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id} in record set {example_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

# If a group field was found, display a boxplot
if example_record_set_id and numeric_field_id and group_field_id:
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load and review a Croissant-compliant dataset with `mlcroissant`, referencing all key components by their `@id`.
- List and extract all record sets, fields, and columns by their `@id`.
- Perform basic EDA operations (filtering, normalization, grouping) directly on referenced data fields by their `@id`.
- Visualize the distribution and grouping of quantitative variables.

This approach supports reproducible, scalable analysis of FAIR datasets with clear provenance and semantic references. For custom analysis, adjust field and record set `@id`s as needed.